<a href="https://colab.research.google.com/github/Nehapriya1423/HTML/blob/main/Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-groq chromadb \
sentence-transformers pypdf langchain-text-splitters -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 739.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/2

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]

print("Uploaded:", pdf_name)

Saving marksheet.pdf to marksheet.pdf
Uploaded: marksheet.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader(pdf_name).load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print("Chunks:", len(chunks))

/tmp/ipykernel_895/3066305709.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Chunks: 20


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(chunks, embeddings)

print("Vector Database Ready!")

/tmp/ipykernel_895/3347663923.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector Database Ready!


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="gsk_API_KEY",
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
def answer_from_pdf(query):
    docs = db.as_retriever().invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
Use ONLY this context to answer.

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)
    return response.content

In [ ]:
def adaptive_answer(question, max_tries=3):
    query = question

    for attempt in range(1, max_tries + 1):
        print(f"\nAttempt {attempt} with query: {query}")

        answer = answer_from_pdf(query)

        print("Answer:", answer)

        if "i don't know" not in answer.lower():
            return answer

        # If the answer is weak, ask the model to rephrase the question
        rephrase_prompt = f"Rephrase this search query differently: {query}"
        query = llm.invoke(rephrase_prompt).content

    return "Could not find an answer after several tries."

In [ ]:
question = input("Ask a question about the PDF: ")

print("\nFinal Answer:")
print(adaptive_answer(question))

Ask a question about the PDF: what is the student name 

Final Answer:

Attempt 1 with query: what is the student name 
Answer: The student name is NEHA PRIYA R.
The student name is NEHA PRIYA R.
